# Problem ID: Category08_Problem01 — Baseline Grid Pursuit-Evasion with Distance-Maximizing Prey
**Course:** CS205B: Artificial Intelligence & Applications | **Unit 4:** Applied Agentic AI & Multi-Agent Learning
**Category 08:** Competitive Pursuit-Evasion (Predator-Prey Grid Dynamics & Simultaneous Policies)

### 1. Problem Scenario:
A security interceptor drone (Predator) pursues an evasive rogue drone (Prey) on a 4x4 airspace sector. A radar-blind static obstacle is located at (1, 1). The Prey actively observes the Predator's position and executes a simultaneous movement to maximize Manhattan distance from the Predator (with a 20% random exploratory jitter).

### 2. System Specifications:
- **Spatial Grid:** 4x4 (16 discrete cells indexed as row * cols + col).
- **Joint State Space (S):** 256 composite states (s = s_predator * 16 + s_prey).
- **Predator Action Space (A):** 4 discrete movements: `0: UP`, `1: DOWN`, `2: LEFT`, `3: RIGHT`.
- **Static Obstacles:** [(1, 1)].
- **Hazardous Traps:** [].
- **Reward Schedule:** Capture = `+15.0`, Step Penalty = `-1.0`, Trap Penalty = `0.0`.

### 3. Student Implementation Deliverables:
1. **Task 1 (Cell 4):** Implement the $\epsilon$-Greedy action selection policy (`choose_action`).
2. **Task 2 (Cell 5):** Implement the Tabular Q-Learning Bellman update rule (`update_agent`):
   $$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \right]$$
3. **Execute & Verify (Cells 6 & 7):** Run the simulation driver and verify convergence of capture reward and steps.


In [ ]:
# Cell 2: Standard Libraries & Environment Parameters (LOCKED)
import numpy as np
import matplotlib.pyplot as plt
import random

# Set deterministic seed for reproducible results
np.random.seed(42)
random.seed(42)

GRID_SIZE = (4, 4)
PREDATOR_START = (0, 0)
PREY_START = (3, 3)
OBSTACLE_STATES = [(1, 1)]
TRAP_STATES = []

NUM_EPISODES = 300
MAX_STEPS_PER_EPISODE = 40
ALPHA = 0.1       # Learning Rate
GAMMA = 0.9       # Discount Factor
EPSILON = 0.2     # Exploration Rate
NUM_STATES = 256
NUM_ACTIONS = 4


In [ ]:
# Cell 3: Pursuit-Evasion Grid Environment Scaffold (LOCKED)
class PursuitEvasionEnv:
    def __init__(self, size=GRID_SIZE):
        self.rows, self.cols = size
        self.num_cells = self.rows * self.cols
        self.reset()

    def reset(self):
        self.predator_pos = list(PREDATOR_START)
        self.prey_pos = list(PREY_START)
        if hasattr(self, 'clock'): self.clock = 0
        if hasattr(self, 'prey_heading'): delattr(self, 'prey_heading')
        return self._get_state_idx(self.predator_pos, self.prey_pos)

    def _get_state_idx(self, p_pos, y_pos):
        p_idx = p_pos[0] * self.cols + p_pos[1]
        y_idx = y_pos[0] * self.cols + y_pos[1]
        return p_idx * self.num_cells + y_idx

    def step(self, action):
        action_deltas = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}

        # 1. Predator deterministic step
        dr, dc = action_deltas.get(action, (0, 0))
        new_pr = min(max(self.predator_pos[0] + dr, 0), self.rows - 1)
        new_pc = min(max(self.predator_pos[1] + dc, 0), self.cols - 1)
        if (new_pr, new_pc) not in OBSTACLE_STATES:
            self.predator_pos = [new_pr, new_pc]

        # 2. Prey simultaneous evasive movement (greedy distance maximization)
        valid_moves = []
        for a in [0, 1, 2, 3]:
            pdr, pdc = action_deltas[a]
            ynr = min(max(self.prey_pos[0] + pdr, 0), self.rows - 1)
            ync = min(max(self.prey_pos[1] + pdc, 0), self.cols - 1)
            if (ynr, ync) not in OBSTACLE_STATES:
                dist = abs(ynr - self.predator_pos[0]) + abs(ync - self.predator_pos[1])
                valid_moves.append((dist, [ynr, ync]))
        if valid_moves:
            valid_moves.sort(key=lambda x: x[0], reverse=True)
            self.prey_pos = random.choice(valid_moves)[1] if (random.random() < 0.20 and len(valid_moves) > 1) else valid_moves[0][1]

        next_state = self._get_state_idx(self.predator_pos, self.prey_pos)
        if self.predator_pos == self.prey_pos:
            return next_state, 15.0, True
        elif tuple(self.predator_pos) in TRAP_STATES:
            return next_state, 0.0, True
        else:
            return next_state, -1.0, False


In [ ]:
# Cell 4: [TASK 1] Agent Class & Action Selection (STUDENT IMPLEMENTATION)
class PursuitEvasionAgent:
    def __init__(self, num_states=NUM_STATES, num_actions=NUM_ACTIONS, alpha=ALPHA, gamma=GAMMA, epsilon=EPSILON):
        self.num_states = num_states
        self.num_actions = num_actions
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.q_table = np.zeros((num_states, num_actions))

    def choose_action(self, state):
        """
        ========================================================================
        TASK 1: Implement Epsilon-Greedy Action Selection Policy.
        - Generate a random float in [0, 1).
        - If float < self.epsilon: return random integer in [0, num_actions - 1].
        - Else: return int(np.argmax(self.q_table[state])).
        ========================================================================
        """
        ### >>> YOUR CODE HERE <<< ###
        pass
        ### ========================== ###


In [ ]:
# Cell 5: [TASK 2] Bellman Q-Learning Update Rule (STUDENT IMPLEMENTATION)
def update_agent(self, state, action, reward, next_state, done):
    """
    ========================================================================
    TASK 2: Implement the Tabular Q-Learning Bellman Update.
    - If done is True: target = reward
    - Else: target = reward + self.gamma * np.max(self.q_table[next_state])
    - Update: self.q_table[state, action] += self.alpha * (target - self.q_table[state, action])
    ========================================================================
    """
    ### >>> YOUR CODE HERE <<< ###
    pass
    ### ========================== ###

PursuitEvasionAgent.update_agent = update_agent


In [ ]:
# Cell 6: Simulation Execution & Test Driver (LOCKED)
env = PursuitEvasionEnv()
agent = PursuitEvasionAgent()

rewards_history = []
steps_history = []

for ep in range(NUM_EPISODES):
    state = env.reset()
    total_reward = 0
    step_count = 0
    done = False

    while not done and step_count < MAX_STEPS_PER_EPISODE:
        action = agent.choose_action(state)
        next_state, reward, done = env.step(action)
        agent.update_agent(state, action, reward, next_state, done)

        state = next_state
        total_reward += reward
        step_count += 1

    rewards_history.append(total_reward)
    steps_history.append(step_count)

print(f"Completed Category08_Problem01 simulation across {NUM_EPISODES} episodes.")
print(f"Average Reward (Last 25 Episodes): {np.mean(rewards_history[-25:]):.2f}")
print(f"Average Steps to Intercept (Last 25 Episodes): {np.mean(steps_history[-25:]):.2f}")


In [ ]:
# Cell 7: Performance Metric Visualization (LOCKED)
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(rewards_history, color='tab:blue', alpha=0.7)
plt.title("Reward Curve — Category08_Problem01")
plt.xlabel("Episode")
plt.ylabel("Cumulative Reward")
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(steps_history, color='tab:orange', alpha=0.7)
plt.title("Pursuit Steps to Capture — Category08_Problem01")
plt.xlabel("Episode")
plt.ylabel("Steps")
plt.grid(True)

plt.tight_layout()
plt.show()
